# MSDS 565 — Unit 1, Notebook 3: Modeling
**Team DataStars**

Trains, evaluates, and compares regressors on the preprocessed dataset from Notebook 2.
Three sklearn regressor types (Linear Regression and Random Forest required, plus one
more) each go through `GridSearchCV`; three Keras architectures go through a manual
5-fold CV loop, and the single best-performing one is retrained and exported. Every
model's CV metrics, test-set metrics, and training time land in one performance table
at the end.


## 1. Setup

### 1.1 Read Notebook 2's split
Reading the exact `processed_train.csv`/`processed_test.csv` Notebook 2 exported --
nothing here is recomputed or re-derived.


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # quiet TensorFlow's info/warning noise

import time
import numpy as np
import pandas as pd
from pathlib import Path

import joblib
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

from tensorflow import keras
from tensorflow.keras import layers, callbacks

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Same local folder Notebook 2 wrote its split to.
DATA_DIR = Path(r"C:\Users\cto\Desktop\MSDS-565\First-Project")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "processed_train.csv")
test_df = pd.read_csv(DATA_DIR / "processed_test.csv")
print(f"train: {train_df.shape}, test: {test_df.shape}")


### 1.2 Defining the target variable
`price` is the regression target. The object-dtype metadata columns Notebook 2
preserved for the Notebook 5 fairness audit get identified the same way Notebook 2
tagged them -- by dtype -- since the raw CSV round-trip naturally leaves only those
columns as `object`; everything modeling-relevant was already numeric or encoded
before export. Both `price` and every metadata column are dropped from `X`, and the
same drop list is applied to `train_df` and `test_df` identically.


In [ ]:
metadata_cols = train_df.select_dtypes(include="object").columns.tolist()
drop_from_X = ["price"] + metadata_cols

X_train = train_df.drop(columns=drop_from_X)
y_train = train_df["price"]
X_test = test_df.drop(columns=drop_from_X)
y_test = test_df["price"]

print(f"Metadata columns excluded from X ({len(metadata_cols)}): {metadata_cols}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")


**What just happened:** `X_train`/`X_test` now hold only numeric, already-scaled
feature columns -- no `price`, no raw categorical metadata. If `X_train.shape[1]`
looks unexpectedly small or large here, it's worth cross-checking against Notebook
2's `feature_cols` count before spending time on GridSearchCV below.


## 2. Models

A results dictionary collects every model's metrics as they're computed, so Section 3 can assemble the final table from one place instead of re-deriving anything.

In [ ]:
results = {}  # model_name -> dict of cv_MAE_mean, cv_MAPE_mean, cv_R2_mean, test_MAE, test_MAPE, test_R2, training_time_sec

def evaluate_on_test(model, name):
    preds = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, preds)
    test_mape = mean_absolute_percentage_error(y_test, preds)
    test_r2 = r2_score(y_test, preds)
    if test_r2 > 0.98:
        print(f"WARNING: {name} test R2 = {test_r2:.4f} -- suspiciously close to 1. "
              f"Double-check Notebook 2 for a feature that leaks price information.")
    return test_mae, test_mape, test_r2


### 2.1 sklearn regressors
Each model below runs one `GridSearchCV(cv=5)` with multi-metric scoring (MAE, MAPE,
R²), refit on MAE. That single fit gives everything the performance table needs: the
winning hyperparameters' mean CV metrics (read directly from `cv_results_` at
`best_index_`), the refit-on-full-training-set model (`best_estimator_`), and the
refit time (`refit_time_`) -- no separate manual refit or second cross-validation
pass required.

**A note on runtime:** with ~240k training rows and a 5-fold grid search, the Random
Forest and Gradient Boosting cells below are the slow part of this notebook --
`n_jobs=-1` is set on all three to use every available core, but budget real time for
these to run, especially the first time.


In [ ]:
scoring = {
    "MAE": "neg_mean_absolute_error",
    "MAPE": "neg_mean_absolute_percentage_error",
    "R2": "r2",
}

def run_grid_search(name, estimator, param_grid):
    grid = GridSearchCV(estimator, param_grid, cv=5, scoring=scoring,
                         refit="MAE", n_jobs=-1)
    grid.fit(X_train, y_train)
    idx = grid.best_index_

    test_mae, test_mape, test_r2 = evaluate_on_test(grid.best_estimator_, name)

    results[name] = {
        "cv_MAE_mean": -grid.cv_results_["mean_test_MAE"][idx],
        "cv_MAPE_mean": -grid.cv_results_["mean_test_MAPE"][idx],
        "cv_R2_mean": grid.cv_results_["mean_test_R2"][idx],
        "test_MAE": test_mae,
        "test_MAPE": test_mape,
        "test_R2": test_r2,
        "training_time_sec": grid.refit_time_,
    }
    print(f"{name} best params: {grid.best_params_}")
    print(f"{name} CV MAE: {results[name]['cv_MAE_mean']:.2f} | test MAE: {test_mae:.2f}")
    return grid


**Linear Regression.** Vanilla OLS doesn't have meaningful hyperparameters to tune
the way tree ensembles do -- this grid only searches `fit_intercept` and the
positive-coefficients constraint, included to satisfy "run a GridSearchCV for every
sklearn model" rather than because a wide search matters here. This is also the
model most likely to struggle: it can't capture the curved price relationships
(horsepower, mileage) the EDA found, and offers no built-in way to handle the
one-hot dummy explosion from Notebook 2 gracefully.


In [ ]:
lr_param_grid = {
    "fit_intercept": [True, False],
    "positive": [True, False],
}
lr_grid = run_grid_search("linreg", LinearRegression(), lr_param_grid)


**Random Forest.** `n_estimators` trades training time for stability; `max_depth`
controls overfitting risk on a dataset this wide (many one-hot dummy columns can let
a tree memorize rare category combinations); `min_samples_leaf` is a second,
gentler regularizer on top of depth. Required by the brief.


In [ ]:
rf_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [None, 15, 30],
    "min_samples_leaf": [1, 4],
}
rf_grid = run_grid_search("rf", RandomForestRegressor(random_state=42), rf_param_grid)


**Gradient Boosting.** The third regressor type -- boosting rather than bagging,
so it should behave differently from the Random Forest above even though both are
tree ensembles. `learning_rate` and `n_estimators` trade off together (slower
learning needs more trees); `max_depth` is kept shallow (2-3) since boosted trees
are usually weak learners by design, unlike Random Forest's deeper, more
independent trees.


In [ ]:
gbr_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3],
}
gbr_grid = run_grid_search("gbr", GradientBoostingRegressor(random_state=42), gbr_param_grid)


**What just happened:** three fitted `GridSearchCV` objects (`lr_grid`, `rf_grid`,
`gbr_grid`), each holding a `best_estimator_` already refit on the full training set.
`results` now has three rows; the keras section below adds a fourth.


### 2.2 Keras networks
Three architectures of increasing complexity, evaluated with a manual 5-fold CV loop
(`KFold`, not `GridSearchCV` -- Keras models aren't natively compatible with sklearn's
grid search). Each architecture gets a fresh model per fold (no weight reuse across
folds), trained with early stopping so a slow-to-converge fold doesn't run the full
epoch budget for no benefit.


In [ ]:
X_train_arr = X_train.to_numpy(dtype="float32")
y_train_arr = y_train.to_numpy(dtype="float32")
X_test_arr = X_test.to_numpy(dtype="float32")
y_test_arr = y_test.to_numpy(dtype="float32")

n_features = X_train_arr.shape[1]

def build_nn1(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def build_nn2(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

def build_nn3(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

architectures = {
    "nn1": build_nn1,   # shallow: one hidden layer -- baseline capacity
    "nn2": build_nn2,   # moderate: two hidden layers + dropout regularization
    "nn3": build_nn3,   # deep: three hidden layers + batch norm + dropout -- most capacity, most regularization
}
print(f"Input dimension: {n_features} features")


**What just happened:** nothing trained yet -- these are just architecture
definitions. `nn1` is the simplest (fewest parameters, least regularization), `nn3`
the most complex. Comparing all three via CV next is what determines whether the
extra capacity in `nn2`/`nn3` actually earns its keep on this data, or just overfits.


Manual 5-fold CV: for each architecture, train and validate on 5 different splits of
the training data, recording MAE/MAPE/R² on the held-out fold each time. This is
what standard `cross_val_score` does under the hood for sklearn models -- written out
explicitly here since Keras needs it done by hand.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

nn_cv_results = {}
for name, builder in architectures.items():
    fold_mae, fold_mape, fold_r2 = [], [], []
    for fold_i, (tr_idx, val_idx) in enumerate(kf.split(X_train_arr), start=1):
        model = builder(n_features)
        model.fit(
            X_train_arr[tr_idx], y_train_arr[tr_idx],
            validation_data=(X_train_arr[val_idx], y_train_arr[val_idx]),
            epochs=50, batch_size=256, callbacks=[early_stop], verbose=0,
        )
        preds = model.predict(X_train_arr[val_idx], verbose=0).ravel()
        fold_mae.append(mean_absolute_error(y_train_arr[val_idx], preds))
        fold_mape.append(mean_absolute_percentage_error(y_train_arr[val_idx], preds))
        fold_r2.append(r2_score(y_train_arr[val_idx], preds))
        print(f"  {name} fold {fold_i}/5 -- MAE: {fold_mae[-1]:.2f}")

    nn_cv_results[name] = {
        "cv_MAE_mean": float(np.mean(fold_mae)),
        "cv_MAPE_mean": float(np.mean(fold_mape)),
        "cv_R2_mean": float(np.mean(fold_r2)),
    }
    print(f"{name} mean CV MAE: {nn_cv_results[name]['cv_MAE_mean']:.2f}\n")

pd.DataFrame(nn_cv_results).T


**What just happened:** 15 models trained total (3 architectures x 5 folds), none of
them kept -- this loop is purely for comparing architectures, matching how
`GridSearchCV` treats each fold's model as disposable. The table above ranks the
three by mean CV MAE; the next cell picks the winner automatically rather than
requiring it to be hand-typed.


**Winner selection and final retrain.** The architecture with the lowest mean CV MAE
gets one more, final training run -- this time on the *entire* training set, no
held-out fold -- since that's the model that actually gets exported and tested.
Everything else built during CV above is discarded, per the brief ("not every
architecture you tried").


In [ ]:
best_nn_name = min(nn_cv_results, key=lambda k: nn_cv_results[k]["cv_MAE_mean"])
print(f"Best Keras architecture: {best_nn_name}")

start = time.time()
best_nn_model = architectures[best_nn_name](n_features)
best_nn_model.fit(X_train_arr, y_train_arr, epochs=50, batch_size=256,
                   callbacks=[early_stop], validation_split=0.1, verbose=0)
nn_training_time = time.time() - start

nn_test_mae, nn_test_mape, nn_test_r2 = evaluate_on_test(best_nn_model, best_nn_name)

results[best_nn_name] = {
    "cv_MAE_mean": nn_cv_results[best_nn_name]["cv_MAE_mean"],
    "cv_MAPE_mean": nn_cv_results[best_nn_name]["cv_MAPE_mean"],
    "cv_R2_mean": nn_cv_results[best_nn_name]["cv_R2_mean"],
    "test_MAE": nn_test_mae,
    "test_MAPE": nn_test_mape,
    "test_R2": nn_test_r2,
    "training_time_sec": nn_training_time,
}
print(f"{best_nn_name} test MAE: {nn_test_mae:.2f}")


## 3. Performance table

One row per model -- three sklearn regressor types plus the single winning Keras architecture -- indexed by model name, exactly as specified in the brief.

In [ ]:
model_metrics = pd.DataFrame(results).T
model_metrics = model_metrics[["cv_MAE_mean", "cv_MAPE_mean", "cv_R2_mean",
                                "test_MAE", "test_MAPE", "test_R2", "training_time_sec"]]
model_metrics


## 4. Discussion

*(Fill in against the actual numbers in the table above once this notebook has run --
placeholders below flag what to look for.)*

- **Which model performs best on the test set, and does that match its CV ranking?**
  A large gap between a model's CV rank and its test rank is itself worth discussing
  -- it usually points to the CV folds not being fully representative, or to
  overfitting that only shows up outside cross-validation.
- **Which model performs worst, and why might that be?** For Linear Regression
  specifically, connect this back to the EDA's Section 6.3 finding that price's
  relationship with horsepower and mileage is curved, not linear -- an OLS model
  structurally cannot capture that no matter how it's tuned.
- **Training time vs. accuracy trade-off** -- is the best-performing model also the
  slowest? If Gradient Boosting or the Keras network wins by only a small margin
  over Random Forest but takes substantially longer to train, that trade-off is
  worth naming explicitly, since Notebook 4's before/after comparison will revisit
  this exact question after feature selection.
- **Any test R² above ~0.98** -- the warning printed automatically in Section 2 if
  this happened for any model. If it did, that model's result shouldn't be trusted
  until Notebook 2 is re-checked for a leaked feature.


## 5. Export

Performance table saved to the local class data folder (not the repo, matching Notebook 2's convention).

In [ ]:
metrics_path = DATA_DIR / "model_metrics.csv"
model_metrics.to_csv(metrics_path)
print(f"Saved performance table -> {metrics_path}")


Every sklearn `best_estimator_` saved to `models/` using the brief's naming
convention, plus the winning Keras model via `model.save(...)`.


In [ ]:
joblib.dump(lr_grid.best_estimator_, MODELS_DIR / "P1_N3_linreg.pkl")
joblib.dump(rf_grid.best_estimator_, MODELS_DIR / "P1_N3_rf.pkl")
joblib.dump(gbr_grid.best_estimator_, MODELS_DIR / "P1_N3_gbr.pkl")
best_nn_model.save(MODELS_DIR / f"P1_N3_{best_nn_name}.keras")

print("Saved models:")
for f in sorted(MODELS_DIR.glob("P1_N3_*")):
    print(" ", f.name, f"({f.stat().st_size / 1e6:.1f} MB)")


**What just happened:** four model files now live in `models/` -- three `.pkl` (one
per sklearn regressor type) and one `.keras` (the single winning architecture, not
all three). Worth a quick check that none of these exceed GitHub's 100 MB file limit
before committing, per the repo requirements -- Random Forest is usually the one to
watch, since `n_estimators=300` with unlimited depth can produce a large file.
